In [1]:
from google.colab import files
uploaded = files.upload()


Saving train.csv to train.csv


In [2]:
!pip install transformers sentencepiece


In [3]:
import pandas as pd

# Load the uploaded file
df = pd.read_csv("train.csv")
df.head()


,gloss,text
0,﻿MEMBERSHIP PARLIAMENT SEE MINUTE\n,﻿membership of parliament see minutes\n
1,APPROVAL MINUTE DESC-PREVIOUS SIT SEE MINUTE\n,approval of minutes of previous sitting see mi...
2,MEMBERSHIP PARLIAMENT SEE MINUTE\n,membership of parliament see minutes\n
3,VERIFICATION CREDENTIALS SEE MINUTE\n,verification of credentials see minutes\n
4,DOCUMENT RECEIVE SEE MINUTE\n,documents received see minutes\n


In [4]:
# Rename for clarity
df = df.rename(columns={"text": "source", "gloss": "target"})

# Optional: Filter out very long or short examples
df = df[df["source"].str.len() < 128]
df = df[df["target"].str.len() < 128]

# Preview
df.head()


,target,source
0,﻿MEMBERSHIP PARLIAMENT SEE MINUTE\n,﻿membership of parliament see minutes\n
1,APPROVAL MINUTE DESC-PREVIOUS SIT SEE MINUTE\n,approval of minutes of previous sitting see mi...
2,MEMBERSHIP PARLIAMENT SEE MINUTE\n,membership of parliament see minutes\n
3,VERIFICATION CREDENTIALS SEE MINUTE\n,verification of credentials see minutes\n
4,DOCUMENT RECEIVE SEE MINUTE\n,documents received see minutes\n


In [10]:
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-base")

# Tokenize
def tokenize(row):
    input_text = "translate English to ASL: " + row['source']
    target_text = row['target']
    input_ids = tokenizer(input_text, padding="max_length", truncation=True, max_length=128, return_tensors="pt").input_ids[0]
    target_ids = tokenizer(target_text, padding="max_length", truncation=True, max_length=128, return_tensors="pt").input_ids[0]
    return input_ids, target_ids

# Apply to a few rows as a test
example_input, example_target = tokenize(df.iloc[0])
print("Input:", tokenizer.decode(example_input))
print("Target:", tokenizer.decode(example_target))


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

Input: translate English to ASL: membership of parliament see minutes</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
Target: MEMBERSHIP PARLIAMENT SEE MINUTE</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><p

In [11]:
from torch.utils.data import Dataset

class ASLTranslationDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.data = dataframe
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        input_ids, target_ids = tokenize(row)
        return {
            "input_ids": input_ids,
            "attention_mask": (input_ids != tokenizer.pad_token_id).long(),
            "labels": target_ids
        }

# Create Dataset instance
train_dataset = ASLTranslationDataset(df, tokenizer)


In [1]:
!pip install sympy==1.12 --upgrade --force-reinstall

  Using cached sympy-1.12-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached sympy-1.12-py3-none-any.whl (5.7 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
  Attempting uninstall: mpmath
    Found existing installation: mpmath 1.3.0
    Uninstalling mpmath-1.3.0:
      Successfully uninstalled mpmath-1.3.0
  Attempting uninstall: sympy
    Found existing installation: sympy 1.12
    Uninstalling sympy-1.12:
      Successfully uninstalled sympy-1.12
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64",

In [2]:
!pip install transformers sentencepiece

In [3]:
import pandas as pd
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from torch.utils.data import Dataset

In [5]:
df = pd.read_csv("train.csv")
df = df.rename(columns={"text": "source", "gloss": "target"})
df = df[df["source"].str.len() < 128]
df = df[df["target"].str.len() < 128]

In [9]:
tokenizer = T5Tokenizer.from_pretrained("t5-base")


In [10]:
def tokenize(row):
    input_text = "translate English to ASL: " + row['source']
    target_text = row['target']
    input_ids = tokenizer(input_text, padding="max_length", truncation=True, max_length=128, return_tensors="pt").input_ids[0]
    target_ids = tokenizer(target_text, padding="max_length", truncation=True, max_length=128, return_tensors="pt").input_ids[0]
    return input_ids, target_ids


In [11]:
class ASLTranslationDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.data = dataframe
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        input_ids, target_ids = tokenize(row)
        return {
            "input_ids": input_ids,
            "attention_mask": (input_ids != tokenizer.pad_token_id).long(),
            "labels": target_ids
        }

train_dataset = ASLTranslationDataset(df, tokenizer)


In [12]:
model = T5ForConditionalGeneration.from_pretrained("t5-base")


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    logging_dir='./logs',
    logging_steps=10,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_steps=500,
    report_to=None  # disables wandb
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

In [ ]:
trainer.train()


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss


Step,Training Loss
